In [1]:
import os
import glob
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm
import matplotlib.pyplot as plt

### # Constants

In [2]:
kps_coco = [
    'nose',
    'left_eye','right_eye',
    'left_ear','right_ear',
    'left_shoulder','right_shoulder',
    'left_elbow','right_elbow',
    'left_wrist','right_wrist',
    'left_hip','right_hip',
    'left_knee','right_knee',
    'left_ankle','right_ankle',
]

In [3]:
len(kps_coco)

17

In [4]:
desc2kp_pairs = {
    'contralateral': [
        ['left_wrist','right_wrist'],
        ['left_ankle','right_ankle'],
        ['left_wrist','right_ankle'],
        ['right_wrist','left_ankle'],
    ],
    'ipsilateral': [
        ['left_wrist','left_ankle'],
        ['right_wrist','right_ankle'],
    ],
    'extension': [
        ['left_wrist','left_shoulder'],
        ['right_wrist','right_shoulder'],
        ['left_ankle','left_hip'],
        ['right_ankle','right_hip'],
    ],
    'contact':[
        ['left_wrist','nose'],
        ['right_wrist','nose'],    
    ]
}

In [5]:
len(desc2kp_pairs)

4

In [6]:
kps_distal = [
    'left_wrist','right_wrist',
    'left_ankle','right_ankle',
]

In [7]:
kps_torso = [
    'left_shoulder','right_shoulder',
    'left_hip','right_hip'
]

### # Functions

In [8]:
def filter_out_diagnoses( df, diagnoses_to_drop ):
    return df[ ~df['diagnosis_singular'].isin(diagnoses_to_drop) ]

In [9]:
def get_distance_between_keypoints( pose, kp1, kp2 ):
    idx1 = kps_coco.index( kp1 )
    idx2 = kps_coco.index( kp2 )
    return np.linalg.norm( (pose[idx1] - pose[idx2]),axis=0 )

In [10]:
def draw_scatterplot( video_stem, clipped=True ):

    row = df_meta_merged[df_meta_merged['video_stem']==video_stem].iloc[0]
    
    start = int(row['gma_video_start_1_fnum'])
    stop  = int(row['gma_video_stop_1_fnum'])

    pose  = np.load( f'./pose_estimate_data_npy_rotated/{video_stem}.npy' ) # n_frames, 17, 2

    if clipped:
        pose_clip  = pose[start:stop]
    else:
        pose_clip  = pose
            
    pose_clip = pose_clip.transpose( (1,2,0) )
        
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(5,5))
    ax.set_facecolor('black')
    
    # DISTAL
    for kp in kps_distal:
        idx = kps_coco.index(kp)
        ax.scatter( x=pose_clip[idx,0,:], y=pose_clip[idx,1,:], label=kp, alpha=0.3, s=4 )
    
    ax.set_xticks([])
    ax.set_yticks([])
    
    ax.legend()
    
    plt.tight_layout()
    plt.show()

### # Load metadata

In [11]:
df_meta_merged = pd.read_csv(
    './df_meta_merged.csv', 
    index_col=0, 
    low_memory=False
)

In [12]:
df_meta_merged['final_code_for_ai_str'].value_counts()#.sum()

final_code_for_ai_str
NKD    1044
DX      409
Name: count, dtype: int64

In [13]:
df_meta_merged['diagnosis_singular'].value_counts()#.sum()

diagnosis_singular
NKD          1044
Other         207
Autism         92
CP             60
Trisomy21      28
SMA            17
Dystro          5
Name: count, dtype: int64

In [14]:
df_meta_merged.shape

(1453, 571)

### # Filter metadata

In [72]:
df_meta_merged_filt = filter_out_diagnoses(df_meta_merged, ['Other','Autism'])

In [73]:
df_meta_merged_filt['diagnosis_singular'].value_counts() #.sum()

diagnosis_singular
NKD          1044
CP             60
Trisomy21      28
SMA            17
Dystro          5
Name: count, dtype: int64

In [74]:
df_meta_merged_filt['final_code_for_ai_str'].value_counts() #.sum()

final_code_for_ai_str
NKD    1044
DX      110
Name: count, dtype: int64

In [75]:
# df_meta_merged_filt.shape

### # Compute Kinematics

In [76]:
# video_stem2pose  = {}
# video_stem2mags  = {}
# video_stem2speed = {}
# video_stem2accel = {}
# video_stem2jerk  = {}

# for i, row in tqdm( df_meta_merged.reset_index().iterrows() ):
# # for i, row in tqdm( df_meta_merged_filt.reset_index().iterrows() ):
    
#     video_stem = row['video_stem']
    
#     start = int(row['gma_video_start_1_fnum'])
#     stop  = int(row['gma_video_stop_1_fnum'])

#     pose  = np.load( f'./pose_estimate_data_npy_rotated/{video_stem}.npy' ).transpose( (1,2,0) ) # 17, 2, n_frames
#     mags  = np.linalg.norm( pose, axis=1) # 17, n_frames
#     speed = np.diff( mags,  axis=-1 ) # 17, n_frames-1
#     accel = np.diff( speed, axis=-1 ) # 17, n_frames-2
#     jerk  = np.diff( accel, axis=-1 ) # 17, n_frames-3
    
#     pose_clip  = pose[:,:,start:stop] # 17, 2, 2700
#     mags_clip  = mags[:,start:stop] # 17, 2700
#     speed_clip = speed[:,start:stop] # '
#     accel_clip = accel[:,start:stop] # '
#     jerk_clip  = jerk[:,start:stop] # '
        
#     video_stem2pose[video_stem]  = pose_clip
#     video_stem2mags[video_stem]  = mags_clip
#     video_stem2speed[video_stem] = speed_clip
#     video_stem2accel[video_stem] = accel_clip
#     video_stem2jerk[video_stem]  = jerk_clip


In [77]:
# video_stem2path_length_stats = {}

# for i, row in tqdm( df_meta_merged.reset_index().iterrows() ):
# # for i, row in tqdm( df_meta_merged_filt.reset_index().iterrows() ):
    
#     video_stem = row['video_stem']
    
#     start = int(row['gma_video_start_1_fnum'])
#     stop  = int(row['gma_video_stop_1_fnum'])

#     pose  = np.load( f'./pose_estimate_data_npy_rotated/{video_stem}.npy' ).transpose( (1,2,0) ) # 17, 2, n_frames
#     mags  = np.linalg.norm( pose, axis=1) # 17, n_frames
#     speed = np.diff( mags,  axis=-1 ) # 17, n_frames-1
    
#     pose_clip  = pose[:,:,start:stop] # 17, 2, 2700
#     mags_clip  = mags[:,start:stop] # 17, 2700
#     speed_clip = speed[:,start:stop] # '

#     path_length_full = np.cumsum( np.abs(speed), axis=-1 )
#     path_length_clip = np.cumsum( np.abs(speed_clip), axis=-1 )

#     path_length_stats    = []
#     path_length_colnames = []
#     for kp in kps_distal:
#         idx = kps_coco.index(kp)
#         path_length_stats.append( path_length_full[idx,-1] )
#         path_length_stats.append( path_length_clip[idx,-1] )
#         path_length_colnames.append( f'path_length_full_{kp}')
#         path_length_colnames.append( f'path_length_clip_{kp}')
    
#     video_stem2path_length_stats[video_stem]  = path_length_stats

In [78]:
# video_stem2tdt_full_stats['585-ca141535-F-72-2010211206']

In [79]:
# video_stem2tdt_clip_stats['585-ca141535-F-72-2010211206']

In [80]:
# df_path_length_stats = pd.DataFrame( video_stem2path_length_stats ).T
# df_path_length_stats.columns = path_length_colnames
# df_path_length_stats['path_length_left']    = df_path_length_stats.filter(like='left').sum(axis=1)
# df_path_length_stats['path_length_right']   = df_path_length_stats.filter(like='right').sum(axis=1)
# df_path_length_stats['path_length_upper']   = df_path_length_stats.filter(like='wrist').sum(axis=1)
# df_path_length_stats['path_length_lower']   = df_path_length_stats.filter(like='ankle').sum(axis=1)
# df_path_length_stats['path_length_LminusR'] = df_path_length_stats['path_length_left'] - df_path_length_stats['path_length_right']
# df_path_length_stats['path_length_UminusL'] = df_path_length_stats['path_length_upper'] - df_path_length_stats['path_length_lower']
# df_path_length_stats['path_length_total']   = df_path_length_stats['path_length_upper'] + df_path_length_stats['path_length_lower']

In [81]:
# df_path_length_stats.sort_values(by='path_length_LminusR').head()

In [82]:
# for video_stem, row in df_path_length_stats.sort_values(by='path_length_LminusR').head().iterrows():
#     draw_scatterplot( video_stem, clipped=False)

In [83]:
# for video_stem, row in df_path_length_stats.sort_values(by='path_length_LminusR').tail().iterrows():
#     draw_scatterplot( video_stem, clipped=False )

In [84]:
# for video_stem, row in df_path_length_stats.sort_values(by='path_length_UminusL').head().iterrows():
#     draw_scatterplot( video_stem, clipped=False)

In [85]:
# for video_stem, row in df_path_length_stats.sort_values(by='path_length_UminusL').tail().iterrows():
#     draw_scatterplot( video_stem, clipped=False)

In [86]:
# for video_stem, row in df_path_length_stats.sort_values(by='path_length_total', ascending=True).head(20).iterrows():
#     draw_scatterplot( video_stem, clipped=False)

In [87]:
# df_meta_merged_with_path_length_stats = \
#     df_meta_merged.merge( df_path_length_stats, left_on='video_stem', right_index=True)

In [88]:
# df_meta_merged_with_path_length_stats['over3m'] = df_meta_merged_with_path_length_stats['age_adjusted'] > 84

In [89]:
# fig, ax = plt.subplots(figsize=(12,6))

# sns.violinplot( 
#     data=df_meta_merged_with_path_length_stats,
#     x='diagnosis_singular',
#     # y='path_length_LminusR',
#     y='path_length_UminusL',
#     hue='over3m',
#     split=True,
#     # color='gray',
#     ax=ax
# )

# # ax.set_xlabel('Diagnosis')

# ax.grid(alpha=0.5)

# plt.tight_layout()
# # plt.savefig('./violinplot_bmi_diagnoses.png')
# plt.show()

### # Scatter Plots

In [90]:
# fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(14,8))

# for i, kp in enumerate( kps_distal ):
    
#     idx = kps_coco.index(kp)

#     axes[0].plot( mags[idx], label=f'{kp}' )
#     if i==len(kps_distal)-1:
#         # axes[0].vlines( x=start, ymin=0, ymax=4.0, color='red' )
#         # axes[0].vlines( x=stop, ymin=0, ymax=4.0, color='red' )
#         axes[0].axvspan( start, stop, color='green', alpha=0.1, label='Annotated Region')
#     axes[1].plot( mags_clip[idx], label=f'{kp}' )

# for ax in axes.flat:
#     ax.legend()
#     ax.grid(alpha=0.5)

# plt.tight_layout()
# plt.show()

In [91]:
# fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(12,8))

# desc = 'extension'
# kp_pairs = desc2kp_pairs[desc]

# for i, (kp1, kp2) in enumerate( kp_pairs ):
    
#     idx1 = kps_coco.index(kp1)
#     idx2 = kps_coco.index(kp2)

#     axes[0].plot( np.linalg.norm( (pose[idx1] - pose[idx2]),axis=0 ), label=f'{kp1}_to_{kp2}' )
#     if i==len(kp_pairs)-1:
#         # axes[0].vlines( x=start, ymin=0, ymax=4.0, color='red' )
#         # axes[0].vlines( x=stop, ymin=0, ymax=4.0, color='red' )
#         axes[0].axvspan( start, stop, color='green', alpha=0.1, label='Annotated Region')
#     axes[1].plot( np.linalg.norm( (pose_clip[idx1] - pose_clip[idx2]),axis=0 ), label=f'{kp1}_to_{kp2}' )

# for ax in axes.flat:
#     ax.legend()
#     ax.grid(alpha=0.5)

# plt.tight_layout()
# plt.show()

In [92]:
# fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(14,8))

# for i, kp in enumerate( kps_distal ):
    
#     idx = kps_coco.index(kp)

#     axes[0].plot( speed[idx], label=f'{kp}' )
#     if i==len(kps_distal)-1:
#         # axes[0].vlines( x=start, ymin=0, ymax=4.0, color='red' )
#         # axes[0].vlines( x=stop, ymin=0, ymax=4.0, color='red' )
#         axes[0].axvspan( start, stop, color='green', alpha=0.1, label='Annotated Region')
#     axes[1].plot( speed_clip[idx], label=f'{kp}' )

# for ax in axes.flat:
#     ax.legend()
#     ax.grid(alpha=0.5)

# plt.tight_layout()
# plt.show()

In [93]:
# fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(14,8))

# for i, kp in enumerate( kps_distal ):
    
#     idx = kps_coco.index(kp)

#     axes[0].plot( accel[idx], label=f'{kp}' )
#     if i==len(kps_distal)-1:
#         # axes[0].vlines( x=start, ymin=0, ymax=4.0, color='red' )
#         # axes[0].vlines( x=stop, ymin=0, ymax=4.0, color='red' )
#         axes[0].axvspan( start, stop, color='green', alpha=0.1, label='Annotated Region')
#     axes[1].plot( accel_clip[idx], label=f'{kp}' )

# for ax in axes.flat:
#     ax.legend()
#     ax.grid(alpha=0.5)

# plt.tight_layout()
# plt.show()

In [94]:
# fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(14,8))

# for i, kp in enumerate( kps_distal ):
    
#     idx = kps_coco.index(kp)

#     axes[0].plot( jerk[idx], label=f'{kp}' )
#     if i==len(kps_distal)-1:
#         # axes[0].vlines( x=start, ymin=0, ymax=4.0, color='red' )
#         # axes[0].vlines( x=stop, ymin=0, ymax=4.0, color='red' )
#         axes[0].axvspan( start, stop, color='green', alpha=0.1, label='Annotated Region')
#     axes[1].plot( jerk_clip[idx], label=f'{kp}' )

# for ax in axes.flat:
#     ax.legend()
#     ax.grid(alpha=0.5)

# plt.tight_layout()
# plt.show()

In [95]:
# draw_scatterplot( video_stem )

### # Windows

In [ ]:
# What should we track over every 5 second interval in the video...

# I don't think we need much from pose directly... extension will be captured in kp-to-kp distances.

# Mean velocity for each distal keypoint
# Std velocity for each distal keypoint - how variable is the velocity
# ??? - how smoothly does velocity change
# Mean acceleration for each distal keypoint
# Mean jerk for each distal keypoint
# Path length for each distal keypoint (TDT)
# Spatial entropy for each distal keypoint - how diverse is the spatial profile
# Temporal entropy for velocity of distal keypoint - how diverse is the kinematic profile
# Distance from proximal joint
# Distance velocity from proximal joint

# Distance between upper extremities (contralateral)
# Distance between lower extremities (contralateral)
# Distance between diagonal extremities (contralateral)
# Distance between same side extremities
# Distance between upper extremities and nose

In [49]:
# def get_arr_windows(
#     arr,
#     fps=30,
#     window_seconds=6,
#     stride_seconds=2,
# ):
    
#     window_size = int(round(window_seconds * fps))

#     if stride_seconds is None:
#         stride_size = window_size
#     else:
#         stride_size = int(round(stride_seconds * fps))

#     # print(arr.shape, fps, window_seconds, stride_seconds, window_size)
    
#     arr_windows = []
#     n_frames = arr.shape[-1]

#     for start in range(0, n_frames - window_size + 1, stride_size):
#         stop = start + window_size
#         if arr.ndim==3:
#             win  = arr[:,:,start:stop]
#         elif arr.ndim==2:
#             win  = arr[:,start:stop]
        
#         arr_windows.append( win )
        
#         # row = {
#         #     'start_frame'    : start,
#         #     'stop_frame'     : stop,
#         #     'start_time_sec' : start / fps,
#         #     'stop_time_sec'  : stop / fps,
#         #     'n_frames'       : window_size,
#         # }

#     return np.asarray(arr_windows)

In [50]:
# pose_windows  = get_arr_windows( pose )

In [51]:
# pose_windows.shape

In [52]:
# mag_windows   = get_arr_windows( mags )

In [53]:
# mag_windows.shape

In [54]:
# speed_windows = get_arr_windows( speed )

In [55]:
# speed_windows.shape

In [56]:
# accel_windows = get_arr_windows( accel )

In [57]:
# jerk_windows  = get_arr_windows( jerk )

In [ ]:
# window_idx = 0

In [58]:
# fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(14,8))

# for kp in kps_distal:
#     idx = kps_coco.index(kp)
#     axes[0,0].plot( mag_windows[window_idx][idx], label=kp )
#     axes[0,1].plot( speed_windows[window_idx][idx], label=kp )
#     axes[1,0].plot( accel_windows[window_idx][idx], label=kp )
#     axes[1,1].plot( jerk_windows[window_idx][idx], label=kp )

# for ax in axes.flat:
#     ax.legend()
#     ax.grid(alpha=0.5)

# plt.tight_layout() 
# plt.show()

<!-- codex-pose-pca-section -->
### # Pose estimate analysis toward clinically relevant classification

This section keeps the exploratory style of the notebook, but makes the PCA pass self-contained and clip-aware. It builds a reusable feature table from the rotated pose arrays for both the full video and the GMA 90 second clip, then visualizes NKD against each `diagnosis_singular` subgroup.


In [96]:
import os
import glob
import numpy as np
import pandas as pd
import plotly.express as px
from pathlib import Path
from IPython.display import Markdown, display
from scipy.stats import kurtosis, skew
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm_notebook

In [97]:
FEATURE_CACHE_PATH = Path('pose_feature_tables/clip_full_pose_features.csv')
FEATURE_CACHE_PATH.parent.mkdir(exist_ok=True)
POSE_DIR = Path('pose_estimate_data_npy_rotated')
FPS = 30

In [ ]:
KPS_COCO = [
    'nose',
    'left_eye', 'right_eye',
    'left_ear', 'right_ear',
    'left_shoulder', 'right_shoulder',
    'left_elbow', 'right_elbow',
    'left_wrist', 'right_wrist',
    'left_hip', 'right_hip',
    'left_knee', 'right_knee',
    'left_ankle', 'right_ankle',
]

In [ ]:
KP_IDX = {kp: idx for idx, kp in enumerate(KPS_COCO)}

In [ ]:
DISTAL_KPS = ['left_wrist', 'right_wrist', 'left_ankle', 'right_ankle']

In [ ]:
GROUPS = {
    'distal': DISTAL_KPS,
    'upper': ['left_wrist', 'right_wrist'],
    'lower': ['left_ankle', 'right_ankle'],
    'left': ['left_wrist', 'left_ankle'],
    'right': ['right_wrist', 'right_ankle'],
}

In [ ]:
PAIR_GROUPS = {
    'contralateral': [
        ('left_wrist', 'right_wrist'),
        ('left_ankle', 'right_ankle'),
        ('left_wrist', 'right_ankle'),
        ('right_wrist', 'left_ankle'),
    ],
    'ipsilateral': [
        ('left_wrist', 'left_ankle'),
        ('right_wrist', 'right_ankle'),
    ],
    'extension': [
        ('left_wrist', 'left_shoulder'),
        ('right_wrist', 'right_shoulder'),
        ('left_ankle', 'left_hip'),
        ('right_ankle', 'right_hip'),
    ],
    'midline_contact': [
        ('left_wrist', 'nose'),
        ('right_wrist', 'nose'),
    ],
}


In [ ]:
def load_pose(video_stem, pose_dir=POSE_DIR):
    pose = np.load(pose_dir / f'{video_stem}.npy')
    if pose.shape[0] == len(KPS_COCO):
        pose = pose.transpose(2, 0, 1)
    return pose.astype(np.float32)


In [ ]:
def safe_clip(pose, start, stop):
    n_frames = pose.shape[0]
    start = 0 if pd.isna(start) else int(max(0, min(start, n_frames - 1)))
    stop = n_frames if pd.isna(stop) else int(max(start + 2, min(stop, n_frames)))
    return pose[start:stop]

In [98]:
def robust_stats(x, prefix):
    x = np.asarray(x, dtype=np.float64).ravel()
    x = x[np.isfinite(x)]
    stat_names = ['mean', 'std', 'median', 'p05', 'p95', 'iqr', 'max', 'skew', 'kurtosis']
    if x.size == 0:
        return {f'{prefix}_{name}': np.nan for name in stat_names}

    p05, p25, p50, p75, p95 = np.percentile(x, [5, 25, 50, 75, 95])
    return {
        f'{prefix}_mean': float(np.mean(x)),
        f'{prefix}_std': float(np.std(x)),
        f'{prefix}_median': float(p50),
        f'{prefix}_p05': float(p05),
        f'{prefix}_p95': float(p95),
        f'{prefix}_iqr': float(p75 - p25),
        f'{prefix}_max': float(np.max(x)),
        f'{prefix}_skew': float(skew(x, bias=False)) if x.size > 2 else np.nan,
        f'{prefix}_kurtosis': float(kurtosis(x, bias=False)) if x.size > 3 else np.nan,
    }

In [99]:
def corr_or_nan(a, b):
    a = np.asarray(a).ravel()
    b = np.asarray(b).ravel()
    mask = np.isfinite(a) & np.isfinite(b)
    if mask.sum() < 5 or np.std(a[mask]) == 0 or np.std(b[mask]) == 0:
        return np.nan
    return float(np.corrcoef(a[mask], b[mask])[0, 1])

In [ ]:
def extract_pose_features(pose, prefix, fps=FPS):
    
    features = {
        f'{prefix}_n_frames': int(pose.shape[0]),
        f'{prefix}_duration_sec': float(pose.shape[0] / fps),
    }

    if pose.shape[0] < 5:
        return features

    velocity = np.diff(pose, axis=0)
    speed = np.linalg.norm(velocity, axis=2)
    accel = np.diff(speed, axis=0)
    jerk = np.diff(accel, axis=0)
    duration = max(speed.shape[0] / fps, 1 / fps)

    for kp in DISTAL_KPS:
        idx = KP_IDX[kp]
        features.update(robust_stats(speed[:, idx], f'{prefix}_speed_{kp}'))
        features.update(robust_stats(np.abs(accel[:, idx]), f'{prefix}_accel_abs_{kp}'))
        features.update(robust_stats(np.abs(jerk[:, idx]), f'{prefix}_jerk_abs_{kp}'))
        features[f'{prefix}_path_rate_{kp}'] = float(np.nansum(speed[:, idx]) / duration)
        features[f'{prefix}_active_fraction_{kp}'] = float(
            np.mean(speed[:, idx] > np.nanpercentile(speed[:, idx], 75))
        )

    for group_name, group_kps in GROUPS.items():
        idxs = [KP_IDX[kp] for kp in group_kps]
        group_speed = speed[:, idxs]
        features.update(robust_stats(group_speed, f'{prefix}_speed_{group_name}'))
        features[f'{prefix}_path_rate_{group_name}'] = float(np.nansum(group_speed) / duration)

    eps = 1e-6
    features[f'{prefix}_path_rate_left_minus_right'] = (
        features[f'{prefix}_path_rate_left'] - features[f'{prefix}_path_rate_right']
    )
    features[f'{prefix}_path_rate_upper_minus_lower'] = (
        features[f'{prefix}_path_rate_upper'] - features[f'{prefix}_path_rate_lower']
    )
    features[f'{prefix}_path_rate_left_right_ratio'] = (
        features[f'{prefix}_path_rate_left'] / (features[f'{prefix}_path_rate_right'] + eps)
    )
    features[f'{prefix}_path_rate_upper_lower_ratio'] = (
        features[f'{prefix}_path_rate_upper'] / (features[f'{prefix}_path_rate_lower'] + eps)
    )

    features[f'{prefix}_wrist_speed_corr'] = corr_or_nan(
        speed[:, KP_IDX['left_wrist']], speed[:, KP_IDX['right_wrist']]
    )
    features[f'{prefix}_ankle_speed_corr'] = corr_or_nan(
        speed[:, KP_IDX['left_ankle']], speed[:, KP_IDX['right_ankle']]
    )
    features[f'{prefix}_left_side_speed_corr'] = corr_or_nan(
        speed[:, KP_IDX['left_wrist']], speed[:, KP_IDX['left_ankle']]
    )
    features[f'{prefix}_right_side_speed_corr'] = corr_or_nan(
        speed[:, KP_IDX['right_wrist']], speed[:, KP_IDX['right_ankle']]
    )

    for pair_group, pairs in PAIR_GROUPS.items():
        pair_distances = []
        for kp1, kp2 in pairs:
            pair_distances.append(
                np.linalg.norm(pose[:, KP_IDX[kp1], :] - pose[:, KP_IDX[kp2], :], axis=1)
            )
        features.update(robust_stats(np.vstack(pair_distances), f'{prefix}_distance_{pair_group}'))

    window_size = int(round(6 * fps))
    stride_size = int(round(2 * fps))
    distal_idxs = [KP_IDX[kp] for kp in DISTAL_KPS]
    if speed.shape[0] >= window_size:
        window_means = []
        for start in range(0, speed.shape[0] - window_size + 1, stride_size):
            window_means.append(np.nanmean(speed[start:start + window_size, distal_idxs]))
        window_means = np.asarray(window_means)
        features[f'{prefix}_window_speed_mean_std'] = float(np.nanstd(window_means))
        features[f'{prefix}_window_speed_mean_cv'] = float(
            np.nanstd(window_means) / (np.nanmean(window_means) + eps)
        )
        features[f'{prefix}_window_speed_p95_p05'] = float(
            np.nanpercentile(window_means, 95) - np.nanpercentile(window_means, 5)
        )

    return features

In [101]:
def build_pose_feature_table(df_meta, use_cache=True):
    if use_cache and FEATURE_CACHE_PATH.exists():
        return pd.read_csv(FEATURE_CACHE_PATH)

    rows = []
    for _, row in tqdm(df_meta.iterrows(), total=len(df_meta)):
        video_stem = row['video_stem']
        if not (POSE_DIR / f'{video_stem}.npy').exists():
            continue

        pose = load_pose(video_stem)
        pose_clip = safe_clip(pose, row['gma_video_start_1_fnum'], row['gma_video_stop_1_fnum'])
        features = {'video_stem': video_stem}
        features.update(extract_pose_features(pose, 'full'))
        features.update(extract_pose_features(pose_clip, 'clip'))
        rows.append(features)

    df_features = pd.DataFrame(rows)
    df_features.to_csv(FEATURE_CACHE_PATH, index=False)
    return df_features

In [ ]:
df_meta_merged = pd.read_csv('./df_meta_merged.csv', index_col=0, low_memory=False)
df_model_base = df_meta_merged[
    df_meta_merged['final_code_for_ai_str'].isin(['NKD', 'DX'])
].copy()

final_code_for_ai_str
NKD    1044
DX      409
Name: count, dtype: int64
diagnosis_singular
NKD          1044
Other         207
Autism         92
CP             60
Trisomy21      28
SMA            17
Dystro          5
Name: count, dtype: int64
(1453, 441)


,video_stem,full_n_frames,full_duration_sec,full_speed_left_wrist_mean,full_speed_left_wrist_std,full_speed_left_wrist_median,full_speed_left_wrist_p05,full_speed_left_wrist_p95,full_speed_left_wrist_iqr,full_speed_left_wrist_max,...,final_code_for_ai_str,diagnosis_singular,pma_at_birth_calc,pma_at_visit_days,age_adjusted,sex,prematurity,gma_video_start_1_fnum,gma_video_stop_1_fnum,nframes_npy
0,585-ca141535-F-72-2010211206,7201,240.033333,0.008894,0.010181,0.005253,0.001016,0.030060,0.007952,0.086634,...,NKD,NKD,273.0,345.0,72.0,0.0,0,2250.0,4950.0,7201.0
1,592-aafcb787-M-28-2010221627,7197,239.900000,0.010034,0.017254,0.003312,0.000377,0.042126,0.009936,0.174236,...,NKD,NKD,266.0,294.0,28.0,1.0,0,2700.0,5400.0,7197.0
2,593-0731adb1-F-34-2010221716,3649,121.633333,0.009178,0.010856,0.005640,0.001008,0.029414,0.008473,0.102764,...,NKD,NKD,266.0,300.0,34.0,0.0,0,949.0,3649.0,3649.0
3,596-6928de1f-M-65-2010261202,6835,227.833333,0.016407,0.023206,0.006534,0.000420,0.065498,0.020535,0.194602,...,DX,Autism,248.0,313.0,33.0,1.0,1,0.0,2700.0,6835.0
4,603-2020-10-26-1755_308c7504_B,3643,121.433333,0.007594,0.010876,0.003714,0.000785,0.026326,0.006708,0.113025,...,NKD,NKD,252.0,424.0,144.0,0.0,1,0.0,2700.0,3643.0


In [ ]:
print(df_model_base['final_code_for_ai_str'].value_counts())
print(df_model_base['diagnosis_singular'].value_counts())

In [ ]:
df_pose_features = build_pose_feature_table(df_model_base, use_cache=True)
df_pose_features = df_pose_features.merge(
    df_model_base[
        [
            'video_stem',
            'final_code_for_ai_str',
            'diagnosis_singular',
            'pma_at_birth_calc',
            'pma_at_visit_days',
            'age_adjusted',
            'sex',
            'prematurity',
            'gma_video_start_1_fnum',
            'gma_video_stop_1_fnum',
            'nframes_npy',
        ]
    ],
    on='video_stem',
    how='left',
)

In [ ]:
print(df_pose_features.shape)
df_pose_features.head()

In [64]:

def run_pose_pca(df, feature_prefix, n_components=5):
    feature_cols = [
        col for col in df.columns
        if col.startswith(f'{feature_prefix}_') and pd.api.types.is_numeric_dtype(df[col])
    ]
    feature_cols = [
        col for col in feature_cols
        if not col.endswith('_n_frames') and not col.endswith('_duration_sec')
    ]

    X = df[feature_cols].replace([np.inf, -np.inf], np.nan)
    pipe = make_pipeline(
        SimpleImputer(strategy='median'),
        StandardScaler(),
        PCA(n_components=n_components, random_state=42),
    )
    pcs = pipe.fit_transform(X)
    pca = pipe.named_steps['pca']

    df_pca = df[
        [
            'video_stem',
            'final_code_for_ai_str',
            'diagnosis_singular',
            'pma_at_birth_calc',
            'pma_at_visit_days',
            'age_adjusted',
            'sex',
            'prematurity',
        ]
    ].copy()
    for idx in range(n_components):
        df_pca[f'PC{idx + 1}'] = pcs[:, idx]

    loadings = pd.DataFrame(
        pca.components_.T,
        index=feature_cols,
        columns=[f'PC{i + 1}' for i in range(n_components)],
    )
    explained = pd.Series(
        pca.explained_variance_ratio_,
        index=[f'PC{i + 1}' for i in range(n_components)],
        name=f'{feature_prefix}_explained_variance_ratio',
    )
    return df_pca, loadings, explained


def plot_pca_overview(df_pca, explained, title):
    fig = px.scatter(
        df_pca,
        x='PC1',
        y='PC2',
        color='diagnosis_singular',
        symbol='final_code_for_ai_str',
        hover_name='video_stem',
        hover_data=['pma_at_visit_days', 'age_adjusted', 'sex', 'prematurity'],
        labels={
            'PC1': f'PC1 ({explained.loc["PC1"]:.1%})',
            'PC2': f'PC2 ({explained.loc["PC2"]:.1%})',
            'diagnosis_singular': 'Diagnosis',
        },
        title=title,
        opacity=0.72,
        width=950,
        height=650,
    )
    fig.update_traces(marker={'size': 7, 'line': {'width': 0.4, 'color': 'white'}})
    fig.update_layout(legend_title_text='Diagnosis', template='plotly_white')
    return fig


def plot_nkd_vs_subgroups(df_pca, explained, title):
    dx_groups = [grp for grp in df_pca['diagnosis_singular'].dropna().unique() if grp != 'NKD']
    facet_frames = []
    for grp in sorted(dx_groups):
        tmp = df_pca[df_pca['diagnosis_singular'].isin(['NKD', grp])].copy()
        tmp['comparison'] = f'NKD vs {grp}'
        tmp['comparison_label'] = np.where(tmp['diagnosis_singular'].eq('NKD'), 'NKD', grp)
        facet_frames.append(tmp)
    df_pairwise = pd.concat(facet_frames, ignore_index=True)

    fig = px.scatter(
        df_pairwise,
        x='PC1',
        y='PC2',
        color='comparison_label',
        facet_col='comparison',
        facet_col_wrap=3,
        hover_name='video_stem',
        hover_data=['diagnosis_singular', 'pma_at_visit_days', 'age_adjusted', 'sex', 'prematurity'],
        labels={
            'PC1': f'PC1 ({explained.loc["PC1"]:.1%})',
            'PC2': f'PC2 ({explained.loc["PC2"]:.1%})',
            'comparison_label': 'Group',
        },
        title=title,
        opacity=0.62,
        width=1150,
        height=750,
    )
    fig.update_traces(marker={'size': 5.5, 'line': {'width': 0.25, 'color': 'white'}})
    fig.update_layout(template='plotly_white', legend_title_text='Group')
    fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
    return fig, df_pairwise


df_pca_clip, loadings_clip, explained_clip = run_pose_pca(df_pose_features, 'clip')
df_pca_full, loadings_full, explained_full = run_pose_pca(df_pose_features, 'full')

pd.concat([explained_clip, explained_full], axis=1).style.format('{:.2%}')


,clip_explained_variance_ratio,full_explained_variance_ratio
PC1,42.25%,41.81%
PC2,9.90%,11.40%
PC3,7.87%,6.97%
PC4,4.09%,3.97%
PC5,3.43%,3.74%


In [65]:

fig_clip_overview = plot_pca_overview(
    df_pca_clip,
    explained_clip,
    'Clip pose features PCA: NKD and diagnosis_singular subgroups',
)
fig_clip_overview.show()

fig_clip_pairwise, df_pca_clip_pairwise = plot_nkd_vs_subgroups(
    df_pca_clip,
    explained_clip,
    'Clip pose features PCA: NKD versus each diagnosis_singular subgroup',
)
fig_clip_pairwise.show()


In [66]:

fig_full_pairwise, df_pca_full_pairwise = plot_nkd_vs_subgroups(
    df_pca_full,
    explained_full,
    'Full-video pose features PCA: NKD versus each diagnosis_singular subgroup',
)
fig_full_pairwise.show()


In [67]:

def show_top_loadings(loadings, pc='PC1', n=12):
    top = (
        loadings[pc]
        .rename('loading')
        .to_frame()
        .assign(abs_loading=lambda x: x['loading'].abs())
        .sort_values('abs_loading', ascending=False)
        .head(n)
    )
    return top[['loading']]

print('Top clip PC1 loadings')
display(show_top_loadings(loadings_clip, 'PC1'))
print('Top clip PC2 loadings')
display(show_top_loadings(loadings_clip, 'PC2'))
print('Top full-video PC1 loadings')
display(show_top_loadings(loadings_full, 'PC1'))
print('Top full-video PC2 loadings')
display(show_top_loadings(loadings_full, 'PC2'))


Top clip PC1 loadings


,loading
clip_speed_distal_mean,0.105517
clip_path_rate_distal,0.105517
clip_speed_distal_iqr,0.103105
clip_speed_distal_p95,0.102560
clip_speed_right_mean,0.102511
clip_path_rate_right,0.102511
clip_path_rate_left,0.102037
clip_speed_left_mean,0.102037
clip_speed_right_iqr,0.099823
clip_speed_distal_median,0.099783


Top clip PC2 loadings


,loading
clip_speed_distal_max,0.158427
clip_speed_left_max,0.148529
clip_speed_right_max,0.141793
clip_speed_upper_max,0.137536
clip_accel_abs_left_ankle_max,0.136131
clip_speed_left_skew,0.132952
clip_speed_lower_max,0.132704
clip_accel_abs_left_ankle_kurtosis,0.132036
clip_speed_distal_skew,0.132005
clip_accel_abs_right_wrist_max,0.131503


Top full-video PC1 loadings


,loading
full_speed_distal_mean,0.105017
full_path_rate_distal,0.105017
full_speed_distal_iqr,0.102933
full_path_rate_right,0.102412
full_speed_right_mean,0.102412
full_speed_distal_p95,0.102197
full_speed_left_mean,0.101994
full_path_rate_left,0.101994
full_speed_right_iqr,0.100131
full_speed_distal_median,0.099587


Top full-video PC2 loadings


,loading
full_speed_distal_max,0.158156
full_speed_lower_max,0.150356
full_accel_abs_left_ankle_max,0.150286
full_speed_left_max,0.149902
full_speed_left_ankle_max,0.146765
full_speed_right_max,0.146493
full_jerk_abs_left_ankle_max,0.143193
full_accel_abs_left_ankle_kurtosis,0.141082
full_accel_abs_left_ankle_skew,0.139579
full_speed_lower_kurtosis,0.137816


<!-- codex-pose-pca-section -->
### # Modeling notes going forward

Descriptive statistics alone are likely too lossy for NKD versus DX because they collapse the structure clinicians care about: temporal organization, symmetry, variability, repertoire, and state changes across the exam. I would treat the feature table above as a transparent baseline, then compare it with time-series models that can preserve sequence structure.

Best next steps:

- Split by infant/record identity before modeling if repeated recordings exist. Avoid random frame/window splits that leak the same infant into train and test.
- Keep a locked test set and use stratified cross-validation inside the training set. Report balanced accuracy, AUROC/AUPRC, sensitivity at clinically acceptable specificity, and subgroup performance by `diagnosis_singular`.
- Adjust for developmental covariates. Age/PMA, prematurity, sex, recording duration, and pose-quality proxies can be confounders; evaluate both movement-only models and movement-plus-covariate models.
- Start interpretable: regularized logistic regression, linear SVM, random forest, gradient boosting, and calibrated probabilities on engineered features. Use permutation importance and subgroup error review before leaning into deep models.
- Investigate clinically meaningful feature families: left-right asymmetry, upper-lower dissociation, distal movement quantity, burstiness, pauses/freezing, variability across 5-10 second windows, limb-pair distance dynamics, synchrony/correlation between limbs, midline hand contact, extension/flexion distance patterns, jerk/smoothness, entropy or sample entropy of distal speed, spectral power/periodicity, and pose-estimator confidence or missingness if available.
- For the 90 second clip versus full video question, compare matched feature families from both windows. If the full video wins, inspect whether the gain is real behavior or just duration/state/context leakage.

Should you use `tsai`? Yes, but as a second-stage benchmark rather than the first modeling tool. It is not overkill if you want MiniRocket/InceptionTime-style sequence classifiers over multichannel pose trajectories or windowed speed signals, especially because MiniRocket is fast and strong for time series. It is overkill for the first pass at clinically explainable features, subgroup PCA, and leakage-resistant validation. My recommendation: build the transparent sklearn feature baseline first, then benchmark `tsai` MiniRocket on standardized channels such as distal x/y coordinates, speeds, pairwise distances, or 6-second windows. Only move to heavier neural architectures if MiniRocket and feature models leave a clear performance gap and you have enough independent infants to validate them.
